## AAA segmentation with SIRE repo

In [1]:
import sys
sys.path.append("../")

### Defining the global controller

In [2]:
import SimpleITK as sitk

from src.sire.inference.totalsegmentator_processor import Roi, TotalSegmentatorProcessor
from src.sire.inference.utils.step_schedulers import ConstantStepScheduler
from src.sire.inference.utils.stopping_criterions import RoiStoppingCriterion, VesselDiameterStoppingCriterion
from src.sire.inference.utils.vessel_config import VesselConfig


class AAAController(TotalSegmentatorProcessor):

    def __call__(self, image_path: str, output_dir: str, seed_path: str = None, device: str = "cpu"):

        # Run the prior total segmentator
        mask, affine = self.run(image_path, output_dir, device)
        _ = sitk.GetArrayFromImage(sitk.ReadImage(image_path))

        # Setup rois
        segments = {
            # Below T12 vertebrae, outside iliac arteries
            "abdominal_aorta": {
                "diameter": {"min": None, "max": None},
                "roi": Roi.intersection(mask, [Roi([32], "below", "max"), Roi([65, 66], "outside")]),
                "step": 1,
            },
            # Below aorta and outside of hips
            "iliac_left": {
                "diameter": {"min": None, "max": 50},
                "roi": Roi.intersection(
                    mask,
                    [Roi([77, 78], "outside"), Roi([52], "below", "min", dilate=20)],
                ),
                "step": 0.5,
            },
            "iliac_right": {
                "diameter": {"min": None, "max": 80},
                "roi": Roi.intersection(
                    mask,
                    [Roi([77, 78], "outside"), Roi([52], "below", "min", dilate=20)],
                ),
                "step": 0.5,
            },
        }

        # Automatically estimate seed points based on rough TotalSegmentator output
        segments["abdominal_aorta"]["seed"] = self._get_seed(mask, affine, segments["abdominal_aorta"]["roi"], 52, p=0.2)
        segments["iliac_right"]["seed"] = self._get_seed(mask, affine, segments["iliac_right"]["roi"], 66, p=0.4)
        segments["iliac_left"]["seed"] = self._get_seed(mask, affine, segments["iliac_left"]["roi"], 65, p=0.4)

        # Return list of vessel configs
        return [
            VesselConfig(
                name=name,
                seed_point=params["seed"],
                segment_every_n_steps=5,
                step_scheduler=ConstantStepScheduler(params["step"]),
                stopping_criterions=[
                    RoiStoppingCriterion(params["roi"]),
                    VesselDiameterStoppingCriterion(
                        max_diameter=params["diameter"]["max"], min_diameter=params["diameter"]["min"]
                    ),
                ],
            )
            for name, params in segments.items()
        ]

### Running SIRE segmentation pipeline

In [ ]:
import os, glob
from tqdm.auto import tqdm

# Import SIRE pipeline
from src.sire.inference.segmentator_tracker import SegmentatorTrackerPipeline

# Import inference models
from src.sire.inference.inference_models import SegmentationInferenceModel, TrackerInferenceModel

# Import pytorch model classes
from src.sire.models.sire_seg import SIRESegmentation
from src.sire.models.sire_tracker import SIRETracker

def inference(
    root_dir: str,
    output_dir: str,
    device: str,
    seeds_dir: str = None
):
    path_dict = {
        path.split("/")[-1].split(".")[0]: path
        for path in glob.glob(os.path.join(root_dir, "**/*.mhd"), recursive=True)
    }

    # Load tracking model
    tracker_model = TrackerInferenceModel(
        model=SIRETracker.load_from_checkpoint("../src/sire/models/checkpoints/tracking_model.ckpt"),
        scales=[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80],
        npoints=32,
        subdivisions=3,
        device=device,
    )

    # Load segmentation model
    segmentation_models = [
        SegmentationInferenceModel(
            model=SIRESegmentation.load_from_checkpoint("../src/sire/models/checkpoints/segmentation_model.ckpt"),
            names=["lumen"],
            scales=[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80],
            npoints=32,
            subdivisions=2,
            device=device,
        ),
    ]

    # Prepare pipeline
    controller = AAAController()
    tracker_pipeline = SegmentatorTrackerPipeline(tracker_model, segmentation_models)

    # Running for each sample
    for sample, image_path in tqdm(path_dict.items(), desc="Samples"):
        os.makedirs(os.path.join(output_dir, sample), exist_ok=True)
        sample_dir = os.path.join(output_dir, sample)

        if seeds_dir is not None:
            seed_path = os.path.join(seeds_dir, f"{sample}_seeds.mrk.json")
        else:
            seed_path = None

        vessel_configs = controller(
            image_path,
            sample_dir,
            seed_path=seed_path,
            device=device if device != "cuda" else "gpu",
        )
        tracker_pipeline.run(
            image_path,
            output_dir=sample_dir,
            vessel_configs=vessel_configs,
            already_tracked_distance=1,
        )

### Run segmentation pipeline over directory

In [ ]:
root_dir = "/Users/patrykrygiel/Documents/UTWENTE/Datasets/Other/Milou/mhd"
output_dir = "/Users/patrykrygiel/Documents/UTWENTE/Datasets/Other/Milou/raw-segmentation"
seeds_dir = "/Users/patrykrygiel/Documents/UTWENTE/Datasets/Other/Milou/seeds"
device = "cpu"

inference(root_dir, output_dir, device, seeds_dir)

### AAA contour meshing pipeline

In [ ]:
from typing import List, Dict
import torch
import numpy as np
import pyvista as pv

from pqdm.processes import pqdm

from src.sire.inr import INR, INRSkeletonPointData
from src.sire.inr.losses import ManifoldLoss, NeuralPullLoss, SkeletonLoss
from src.sire.inr.models import Siren
from src.sire.reconstruct.vascular_model import VascularModel


def euler_characteristic(poly: pv.PolyData):
    return poly.n_points - poly.extract_all_edges().n_lines + poly.n_faces_strict

def reconstruct(name: str, out_dir: str, contour: np.array, centerline: np.array, omega: int):
    losses = [(NeuralPullLoss(), 0.15), (ManifoldLoss(), 0.05), (SkeletonLoss(), 0.1)]

    os.makedirs(os.path.join(out_dir, "models"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "meshes"), exist_ok=True)

    if len(contour.reshape(-1, 3)) < 500:
        print(f"{os.path.join(out_dir, name)}: Invalid contour")
        return 1

    inr_point_data = INRSkeletonPointData(
        torch.tensor(centerline), torch.tensor(contour.reshape(-1, 3)), norm_scale=1.6
    )
    inr_module = INR(Siren([3, 64, 64, 64, 64, 64, 64, 1], omega=omega), losses=losses, device="cpu")
    inr_module.fit(inr_point_data, n_points=1000, n_iters=5_000, verbose=False, plots=False)
    inr_module.export_to_onnx((1000, 3), os.path.join(out_dir, "models", f"{name}_{omega}.onnx"))

    poly, sdf = inr_point_data.reconstruct_pyvista(
        os.path.join(out_dir, "models", f"{name}_{omega}.onnx"), resolution=256, verbose=False
    )

    if euler_characteristic(poly) == 2:
        poly.save(sample := os.path.join(out_dir, "meshes", f"{name}_{omega}.vtp"))
        print(f"Correct topology - saving sample: {sample}")

    return name, omega, euler_characteristic(poly)

In [ ]:
def reconstruct_directory(in_dir: str, out_dir: str, omegas: List[int], pruning: Dict[str, int], n_jobs: int = 1):
    args = []
    samples = [filename for filename in sorted(os.listdir(in_dir)) if filename != ".DS_Store"]

    for sample in samples:
        branches = [
            "_".join(filename.split("_")[1:]).split(".")[0]
            for filename in sorted(os.listdir(os.path.join(in_dir, sample, "contour", "lumen")))
            if filename != ".DS_Store"
        ]

        # Load vascular model from contour directory
        vascular_model = VascularModel.load_from_directory(
            os.path.join(in_dir, sample, "contour", "lumen"), filenames=branches
        )

        # Correct centerlines
        bifurcation_point = vascular_model.correct_bifurcation(
            "abdominal_aorta", "iliac_left", "iliac_right", merge_tol=3
        )

        # Get pruned contours
        contours = {
            name: vascular_model.get_pruned_contour(name, "abdominal_aorta", pruning[name])
            for name in branches
            if name != "abdominal_aorta"
        }
        contours["abdominal_aorta"] = (
            vascular_model.contours["abdominal_aorta"],
            vascular_model.contours["abdominal_aorta"].mean(axis=1),
        )

        # Save corrected centerlines
        for name in vascular_model.centerlines.keys():
            os.makedirs(os.path.join(out_dir, sample, "centerlines"), exist_ok=True)

            if "iliac" in name:
                centerline = vascular_model.get_corrected_contour(name, connect_to=bifurcation_point).mean(axis=1)
            else:
                centerline = vascular_model.get_corrected_contour(name).mean(axis=1)

            pv.PolyData(
                centerline, lines=np.array([[2, i, i + 1] for i in range(len(centerline) - 1)]).flatten().astype(int)
            ).save(os.path.join(out_dir, sample, "centerlines", f"{name}.vtp"))

        # Add segments for reconstruction
        args.extend(
            [
                (
                    "full",
                    os.path.join(out_dir, sample),
                    np.concatenate([contour[0] for _, contour in contours.items()]),
                    np.concatenate([contour[1] for _, contour in contours.items()]),
                    omega,
                    False,
                )
                for omega in omegas
            ]
        )

    pqdm(args, reconstruct, argument_type="args", n_jobs=n_jobs)

### Run meshing pipeline over directory

In [ ]:
n_jobs = 1
omegas = [12, 14, 16, 18, 20]
pruning = {"iliac_left": 8, "iliac_right": 8}

input_dir = "/Users/patrykrygiel/Documents/UTWENTE/Datasets/Other/test/raw-segmentation"
output_dir = "/Users/patrykrygiel/Documents/UTWENTE/Datasets/Other/test/raw-reconstruction"

reconstruct_directory(
    in_dir=input_dir, 
    out_dir=output_dir, 
    omegas=omegas, 
    pruning=pruning, 
    n_jobs=n_jobs
)